# Practical 5: Hyperparameter Tuning and Model Optimization

**Problem Statement:** Improve deep learning model performance through tuning learning rate, batch size, number of layers, and activation functions.

**Activities:**
1. Experiment with optimizers
2. Tune model parameters
3. Compare validation accuracy

**Dataset:** Breast Cancer Wisconsin dataset - numeric cell-nuclei features, binary malignant/benign target.

## 1. Import Libraries and Load Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
df = data.frame

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (455, 30) Test shape: (114, 30)


## 2. Reusable Model Builder

A single function builds and trains a model for a given combination of hyperparameters, so each experiment only varies one setting at a time.

In [2]:
def build_and_train(optimizer='adam', learning_rate=0.001, num_layers=2,
                     units=32, activation='relu', batch_size=16, epochs=30):
    layers = [keras.layers.Input(shape=(X_train.shape[1],))]
    for _ in range(num_layers):
        layers.append(keras.layers.Dense(units, activation=activation))
    layers.append(keras.layers.Dense(1, activation='sigmoid'))

    model = keras.Sequential(layers)

    opt_map = {
        'adam': keras.optimizers.Adam(learning_rate=learning_rate),
        'sgd': keras.optimizers.SGD(learning_rate=learning_rate),
        'rmsprop': keras.optimizers.RMSprop(learning_rate=learning_rate)
    }

    model.compile(optimizer=opt_map[optimizer], loss='binary_crossentropy', metrics=['accuracy'])

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    val_accuracy = history.history['val_accuracy'][-1]
    return history, val_accuracy

## 3. Experiment 1: Optimizers

In [3]:
optimizer_results = []
for opt in ['adam', 'sgd', 'rmsprop']:
    _, val_acc = build_and_train(optimizer=opt)
    optimizer_results.append({'Optimizer': opt, 'Validation Accuracy': val_acc})

pd.DataFrame(optimizer_results)

,Optimizer,Validation Accuracy
0,adam,1.000000
1,sgd,0.879121
2,rmsprop,1.000000


## 4. Experiment 2: Learning Rate

In [ ]:
lr_results = []
for lr in [0.1, 0.01, 0.001, 0.0001]:
    _, val_acc = build_and_train(learning_rate=lr)
    lr_results.append({'Learning Rate': lr, 'Validation Accuracy': val_acc})

pd.DataFrame(lr_results)

## 5. Experiment 3: Batch Size

In [ ]:
batch_results = []
for bs in [8, 16, 32, 64]:
    _, val_acc = build_and_train(batch_size=bs)
    batch_results.append({'Batch Size': bs, 'Validation Accuracy': val_acc})

pd.DataFrame(batch_results)

## 6. Experiment 4: Number of Layers

In [ ]:
layer_results = []
for n in [1, 2, 3, 4]:
    _, val_acc = build_and_train(num_layers=n)
    layer_results.append({'Num Layers': n, 'Validation Accuracy': val_acc})

pd.DataFrame(layer_results)

## 7. Experiment 5: Activation Function

In [ ]:
activation_results = []
for act in ['relu', 'tanh', 'sigmoid', 'elu']:
    _, val_acc = build_and_train(activation=act)
    activation_results.append({'Activation': act, 'Validation Accuracy': val_acc})

pd.DataFrame(activation_results)

## 8. Best Configuration

In [ ]:
best_optimizer = max(optimizer_results, key=lambda r: r['Validation Accuracy'])['Optimizer']
best_lr = max(lr_results, key=lambda r: r['Validation Accuracy'])['Learning Rate']
best_batch = max(batch_results, key=lambda r: r['Validation Accuracy'])['Batch Size']
best_layers = max(layer_results, key=lambda r: r['Validation Accuracy'])['Num Layers']
best_activation = max(activation_results, key=lambda r: r['Validation Accuracy'])['Activation']

print("Best optimizer:", best_optimizer)
print("Best learning rate:", best_lr)
print("Best batch size:", best_batch)
print("Best number of layers:", best_layers)
print("Best activation function:", best_activation)

history_final, val_acc_final = build_and_train(
    optimizer=best_optimizer, learning_rate=best_lr,
    num_layers=best_layers, activation=best_activation,
    batch_size=best_batch
)
print("\nFinal tuned model validation accuracy:", val_acc_final)

## Conclusion

In this practical, we:
- Built a reusable training function to isolate one hyperparameter at a time
- Compared optimizers (Adam, SGD, RMSprop)
- Tuned learning rate, batch size, number of layers, and activation function
- Combined the best-performing settings into a final tuned model